# Dataset 1 — Appliances Energy Prediction (UCI)

In [1]:
import pandas as pd

df = pd.read_csv('energydata_complete.csv')
colunas = ['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3']
df = df[colunas]
df = df.sample(frac=0.10, random_state=42).reset_index(drop=True)

## 1. Exploração inicial

In [2]:
df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3
0,2016-03-14 01:40:00,40,0,20.890000,35.400000,17.760000,39.163333,20.290000,36.900000
1,2016-01-30 20:00:00,90,10,21.890000,53.100000,21.290000,45.360000,21.633333,49.226667
2,2016-03-15 03:00:00,50,0,21.390000,35.500000,17.633333,40.530000,21.666667,35.200000
3,2016-04-20 10:10:00,50,0,21.390000,41.033333,23.890000,34.840000,22.033333,36.933333
4,2016-03-13 08:10:00,70,0,19.963333,35.126667,16.463333,40.126667,20.000000,36.400000


In [3]:
df.shape

(1974, 9)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1974 entries, 0 to 1973
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1974 non-null   object 
 1   Appliances  1974 non-null   int64  
 2   lights      1974 non-null   int64  
 3   T1          1974 non-null   float64
 4   RH_1        1974 non-null   float64
 5   T2          1974 non-null   float64
 6   RH_2        1974 non-null   float64
 7   T3          1974 non-null   float64
 8   RH_3        1974 non-null   float64
dtypes: float64(6), int64(2), object(1)
memory usage: 138.9+ KB


In [5]:
df.describe()

,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3
count,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000,1974.000000
mean,93.044580,3.839919,21.664702,40.219194,20.293635,40.411386,22.231615,39.209733
std,93.826149,7.937076,1.574883,3.971923,2.126958,4.037685,1.957275,3.288818
min,20.000000,0.000000,16.790000,27.733333,16.100000,24.063333,17.200000,30.133333
25%,50.000000,0.000000,20.700000,37.205833,18.790000,37.760000,20.700000,36.900000
50%,60.000000,0.000000,21.600000,39.590000,20.000000,40.433333,22.100000,38.433333
75%,100.000000,0.000000,22.600000,42.991667,21.500000,43.228333,23.290000,41.741250
max,770.000000,50.000000,26.260000,57.496667,28.917500,56.026667,29.198571,49.226667


## 2. Renomear colunas

In [6]:
df = df.rename(columns={
    'Appliances': 'Consumo_Eletrodomesticos',
    'T1': 'Temp_Cozinha',
    'RH_1': 'Umidade_Cozinha',
    'T2': 'Temp_Sala',
    'RH_2': 'Umidade_Sala',
    'T3': 'Temp_Lavanderia',
    'RH_3': 'Umidade_Lavanderia'
})

df.columns

Index(['date', 'Consumo_Eletrodomesticos', 'lights', 'Temp_Cozinha',
       'Umidade_Cozinha', 'Temp_Sala', 'Umidade_Sala', 'Temp_Lavanderia',
       'Umidade_Lavanderia'],
      dtype='object')

## 3. Maior consumo registrado

In [7]:
maior_consumo = df['Consumo_Eletrodomesticos'].max()
maior_consumo

770

## 4. Limiar de 70% do máximo e filtragem

In [8]:
limiar_consumo = maior_consumo * 0.70
limiar_consumo

539.0

In [9]:
df_alto_consumo = df[df['Consumo_Eletrodomesticos'] > limiar_consumo]
df_alto_consumo.head()

,date,Consumo_Eletrodomesticos,lights,Temp_Cozinha,Umidade_Cozinha,Temp_Sala,Umidade_Sala,Temp_Lavanderia,Umidade_Lavanderia
352,2016-05-26 17:10:00,620,0,24.390000,44.333333,25.370000,37.736000,26.730000,38.863333
427,2016-03-19 11:20:00,650,0,20.890000,36.700000,18.290000,39.663333,21.200000,36.000000
453,2016-03-21 18:40:00,560,30,21.356667,36.090000,19.566667,37.163333,21.633333,35.030000
616,2016-05-14 17:50:00,570,10,24.890000,32.666667,23.560000,31.140000,23.600000,30.823333
696,2016-01-21 19:10:00,590,30,19.856667,47.663333,18.700000,34.100000,18.426667,37.833333


## 5. Contagem e percentual

In [10]:
qtd_alto_consumo = df_alto_consumo.shape[0]
qtd_total = df.shape[0]
percentual_alto_consumo = (qtd_alto_consumo / qtd_total) * 100

print(f'Registros de alto consumo: {qtd_alto_consumo}')
print(f'Total de registros: {qtd_total}')
print(f'Percentual: {percentual_alto_consumo:.2f}%')

Registros de alto consumo: 22
Total de registros: 1974
Percentual: 1.11%


## 6. Temperatura média (T1) e filtro combinado

In [11]:
temp_media = df['Temp_Cozinha'].mean()
temp_media

np.float64(21.664702480734395)

In [12]:
df_alto_consumo_e_temp = df[
    (df['Consumo_Eletrodomesticos'] > limiar_consumo) &
    (df['Temp_Cozinha'] > temp_media)
]

df_alto_consumo_e_temp.head()

,date,Consumo_Eletrodomesticos,lights,Temp_Cozinha,Umidade_Cozinha,Temp_Sala,Umidade_Sala,Temp_Lavanderia,Umidade_Lavanderia
352,2016-05-26 17:10:00,620,0,24.390000,44.333333,25.370000,37.736000,26.73,38.863333
616,2016-05-14 17:50:00,570,10,24.890000,32.666667,23.560000,31.140000,23.60,30.823333
916,2016-02-11 18:50:00,750,30,22.430000,43.000000,21.926667,40.133333,21.23,42.596667
993,2016-02-09 19:10:00,690,10,22.533333,45.026667,22.066667,39.790000,22.39,41.930000
1182,2016-05-13 11:10:00,600,20,24.890000,48.133333,28.917500,37.947500,27.10,42.440000


In [13]:
qtd_combinado = df_alto_consumo_e_temp.shape[0]
percentual_combinado = (qtd_combinado / qtd_total) * 100

print(f'Registros com alto consumo E temperatura acima da média: {qtd_combinado}')
print(f'Percentual: {percentual_combinado:.2f}%')

Registros com alto consumo E temperatura acima da média: 8
Percentual: 0.41%


## 7. Comparação entre os dois DataFrames

In [14]:
print(f'Somente alto consumo: {qtd_alto_consumo} registros ({percentual_alto_consumo:.2f}%)')
print(f'Alto consumo + temperatura acima da média: {qtd_combinado} registros ({percentual_combinado:.2f}%)')

Somente alto consumo: 22 registros (1.11%)
Alto consumo + temperatura acima da média: 8 registros (0.41%)
